In [1]:
import sys, os
os.chdir('..')  # set CWD to project root
sys.path.insert(0, 'src')

import pandas as pd

from etl import *
from regression import *
from sims_engine import *

In [2]:
pd.set_option('display.max_columns', None)

In [3]:
%load_ext autoreload
%autoreload 2

In [4]:
CONFIG = {
    'elo_date': '2026-02-19',  # date for elo ratings from clubelo.com
    'opta_date': '2026-02-26',  # date for elo ratings from the opta -> clubelo regression; elo_date from day X is before the games are played, for opta it depends
    'number_of_sims': 5000,
    'league_id': 109,
    'season': 2025,
    'head_size': 36,
    'country_code_elo': None,  # use this attr. to use elo ratings from clubelo.com
    'country_code_api': 'POL',  # use this attr. to use elo ratings from the opta -> clubelo regression
    'stdev': 0,
    'update_fixtures': False,
    'is_european_league': False,
    'round_no': 28,
    'point_deductions': {'Lechia Gdansk': 5},
    'games_to_overwrite': {
        ('Kalisz', 'Jastrzębie'): (3, 0),
        ('Jastrzębie', 'Unia Skierniewice'): (0, 3),
        ('Podbeskidzie', 'Jastrzębie'): (3, 0),
        ('Jastrzębie', 'Rekord Bielsko-Biała'): (0, 3),
        ('Stal Stalowa Wola', 'Jastrzębie'): (3, 0),
        ('Jastrzębie', 'Śląsk Wrocław II'): (0, 3),
    }
}
code = CONFIG['country_code_elo'] if CONFIG['country_code_elo'] is not None else CONFIG['country_code_api']
CONFIG['sorting_order'] = get_sorting_order_for_country_code(code)

In [5]:
# download_elo_data(CONFIG['elo_date'])

In [6]:
# main_regression(**CONFIG)

In [7]:
standings_df, fixtures_matrix = build_historical_standings_table_after_at_most_n_rounds(**CONFIG)
standings_df.head(CONFIG['head_size'])

,Club,Elo,Matches played,Wins,Draws,Losses,Goals for,Goals against,Goal difference,Goals away,Points,Random order,H2H
1,Unia Skierniewice,1066.29,28,17,5,6,56,38,18,25,56,3,1
2,Olimpia Grudziądz,1066.29,28,15,8,5,57,33,24,27,53,6,1
3,Warta Poznań,1102.42,28,14,10,4,46,31,15,19,52,12,1
4,Podhale Nowy Targ,1021.13,28,11,12,5,37,27,10,15,45,5,1
5,Sandecja Nowy Sącz,998.55,28,11,11,6,43,35,8,20,44,4,1
6,Śląsk Wrocław II,1012.10,28,12,7,9,50,38,12,29,43,10,1
7,Podbeskidzie,1023.39,28,12,6,10,51,41,10,25,42,7,1
8,Chojniczanka Chojnice,1059.52,28,11,8,9,45,38,7,24,41,14,1
9,Świt Skolwin,1030.16,28,11,7,10,45,48,-3,19,40,8,1
10,Hutnik Kraków,987.26,28,10,8,10,41,35,6,17,38,1,1


In [8]:
CONFIG['update_fixtures'] = False

In [9]:
standings_df['Elo'].mean()

1017.868888888889

In [10]:
float(round(standings_df['Points'].sum() / standings_df['Matches played'].sum(), 2))

1.35

In [11]:
sample_season = simulate_season_after_n_rounds(**CONFIG, standings_df=standings_df)
sample_season.head(CONFIG['head_size'])

,Club,Elo,Matches played,Wins,Draws,Losses,Goals for,Goals against,Goal difference,Goals away,Points,Random order,H2H
1,Unia Skierniewice,1066.29,34,22,6,6,68,39,29,32,72,16,1
2,Warta Poznań,1102.42,34,16,12,6,52,37,15,24,60,12,1
3,Olimpia Grudziądz,1066.29,34,16,10,8,61,41,20,29,58,10,1
4,Podbeskidzie,1023.39,34,17,7,10,63,42,21,29,58,13,2
5,Śląsk Wrocław II,1012.10,34,15,9,10,59,42,17,34,54,5,1
6,Sandecja Nowy Sącz,998.55,34,12,14,8,48,42,6,23,50,4,1
7,Podhale Nowy Targ,1021.13,34,12,13,9,40,36,4,17,49,17,1
8,Rekord Bielsko-Biała,994.03,34,13,10,11,49,46,3,28,49,6,2
9,Chojniczanka Chojnice,1059.52,34,13,9,12,50,45,5,26,48,9,1
10,Resovia Rzeszów,1041.45,34,11,12,11,46,44,2,23,45,11,1


In [12]:
float(round(sample_season['Points'].sum() / sample_season['Matches played'].sum(), 2))

1.35

In [13]:
simulate_odds(**CONFIG, standings_df=standings_df).head(CONFIG['head_size'])

,Home Team,Away Team,Odds H,Odds D,Odds A
0,Sandecja Nowy Sącz,Kalisz,2.29,3.54,3.55
1,Sokół Kleczew,Olimpia Grudziądz,2.78,3.42,2.87
2,Jastrzębie,Zaglebie Sosnowiec,2.85,3.42,2.80
3,Rekord Bielsko-Biała,Resovia Rzeszów,2.68,3.43,2.98
4,Chojniczanka Chojnice,Stal Stalowa Wola,2.33,3.52,3.49
5,Śląsk Wrocław II,Unia Skierniewice,2.74,3.43,2.92
6,ŁKS Łódź II,Podhale Nowy Targ,3.04,3.44,2.63
7,Świt Skolwin,Hutnik Kraków,2.11,3.67,3.94
8,Warta Poznań,Podbeskidzie,1.94,3.87,4.44


In [14]:
# full table sim
results, h2h_table = run_full_table_sims(**CONFIG, standings_df=standings_df, fixtures_matrix=fixtures_matrix)
results.head(CONFIG['head_size'])

100%|██████████| 5000/5000 [02:12<00:00, 37.86it/s]

5000 simulations


,Club,Elo,xPts,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18
1,Unia Skierniewice,1066.29,66.92,82.2,15.7,2.1,0.1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,Olimpia Grudziądz,1066.29,61.84,10.9,42.7,40.9,4.5,0.8,0.2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,Warta Poznań,1102.42,61.31,6.9,40.4,44.6,5.9,1.7,0.4,0.1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,Śląsk Wrocław II,1012.10,53.39,0.0,0.2,3.6,29.0,24.5,17.8,12.3,7.1,3.5,1.4,0.6,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,Podhale Nowy Targ,1021.13,52.89,0.0,0.6,4.4,21.8,20.6,17.3,13.3,11.0,6.6,3.4,0.9,0.1,0.0,0.0,0.0,0.0,0.0,0.0
6,Podbeskidzie,1023.39,52.04,0.0,0.1,1.4,16.2,19.8,19.4,16.9,12.2,7.5,3.8,1.8,0.7,0.2,0.0,0.0,0.0,0.0,0.0
7,Sandecja Nowy Sącz,998.55,51.44,0.0,0.2,2.3,13.1,16.2,17.5,17.9,13.7,10.1,5.5,2.6,0.8,0.1,0.0,0.0,0.0,0.0,0.0
8,Chojniczanka Chojnice,1059.52,49.83,0.0,0.0,0.5,4.6,8.8,14.1,16.8,19.9,14.9,9.3,6.0,3.4,1.4,0.3,0.1,0.0,0.0,0.0
9,Świt Skolwin,1030.16,48.50,0.0,0.0,0.3,4.4,5.6,8.3,11.1,14.6,15.7,15.4,11.1,7.5,4.6,1.4,0.1,0.0,0.0,0.0
10,Hutnik Kraków,987.26,45.33,0.0,0.0,0.0,0.3,0.9,2.2,4.1,6.2,10.6,13.7,16.3,17.6,15.3,10.1,2.7,0.0,0.0,0.0


In [15]:
CONFIG['games_to_overwrite'][('Unia Skierniewice', 'Sokół Kleczew')] = (2, 0)
results, h2h_table = run_full_table_sims(**CONFIG, standings_df=standings_df, fixtures_matrix=fixtures_matrix)
results.head(CONFIG['head_size'])

100%|██████████| 5000/5000 [02:02<00:00, 40.87it/s]

5000 simulations


,Club,Elo,xPts,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18
1,Unia Skierniewice,1066.29,68.25,90.8,8.5,0.7,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,Olimpia Grudziądz,1066.29,61.86,6.1,46.7,41.7,4.5,0.9,0.1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,Warta Poznań,1102.42,61.34,3.1,43.8,46.1,5.3,1.4,0.3,0.1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,Śląsk Wrocław II,1012.10,53.33,0.0,0.2,3.1,29.5,24.1,17.6,12.0,7.4,4.0,1.6,0.5,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,Podhale Nowy Targ,1021.13,52.91,0.0,0.5,3.6,22.7,21.2,17.1,13.3,10.4,6.6,3.2,1.2,0.1,0.0,0.0,0.0,0.0,0.0,0.0
6,Podbeskidzie,1023.39,52.09,0.0,0.1,1.5,15.9,20.3,19.9,17.2,11.7,7.6,3.4,1.7,0.6,0.1,0.0,0.0,0.0,0.0,0.0
7,Sandecja Nowy Sącz,998.55,51.45,0.0,0.2,2.6,12.9,15.6,17.6,17.5,14.7,10.0,5.7,2.3,0.7,0.1,0.0,0.0,0.0,0.0,0.0
8,Chojniczanka Chojnice,1059.52,49.89,0.0,0.0,0.6,4.6,9.0,12.8,17.8,20.3,15.1,10.0,5.6,2.8,0.9,0.2,0.0,0.0,0.0,0.0
9,Świt Skolwin,1030.16,48.53,0.0,0.0,0.1,3.9,5.8,9.6,11.1,14.3,16.6,14.9,11.2,7.1,3.7,1.4,0.2,0.0,0.0,0.0
10,Hutnik Kraków,987.26,45.40,0.0,0.0,0.1,0.4,1.0,2.1,3.9,6.3,10.9,15.3,17.8,16.4,15.7,7.5,2.6,0.0,0.0,0.0


In [16]:
CONFIG['games_to_overwrite'][('Unia Skierniewice', 'Sokół Kleczew')] = (1, 1)
results, h2h_table = run_full_table_sims(**CONFIG, standings_df=standings_df, fixtures_matrix=fixtures_matrix)
results.head(CONFIG['head_size'])

100%|██████████| 5000/5000 [01:03<00:00, 78.81it/s]

5000 simulations


,Club,Elo,xPts,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18
1,Unia Skierniewice,1066.29,66.24,79.1,18.0,2.9,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,Olimpia Grudziądz,1066.29,61.87,12.8,41.7,40.3,4.1,0.8,0.3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,Warta Poznań,1102.42,61.30,8.1,39.1,45.4,5.6,1.3,0.4,0.1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,Śląsk Wrocław II,1012.10,53.30,0.0,0.4,2.8,29.3,23.9,18.2,11.5,8.1,3.9,1.3,0.5,0.1,0.0,0.0,0.0,0.0,0.0,0.0
5,Podhale Nowy Targ,1021.13,52.88,0.0,0.4,3.7,22.7,20.8,17.9,13.9,10.1,7.2,2.4,0.8,0.2,0.1,0.0,0.0,0.0,0.0,0.0
6,Podbeskidzie,1023.39,52.02,0.0,0.1,1.4,16.4,19.6,19.3,17.1,12.2,7.4,3.9,1.9,0.6,0.2,0.0,0.0,0.0,0.0,0.0
7,Sandecja Nowy Sącz,998.55,51.44,0.0,0.4,2.6,12.6,16.1,16.9,18.4,14.3,10.2,5.4,2.3,0.7,0.2,0.0,0.0,0.0,0.0,0.0
8,Chojniczanka Chojnice,1059.52,49.84,0.0,0.0,0.5,5.1,9.6,13.1,15.7,19.5,15.7,9.6,5.9,3.5,1.2,0.5,0.0,0.0,0.0,0.0
9,Świt Skolwin,1030.16,48.53,0.0,0.0,0.3,3.8,6.4,8.9,12.3,14.0,15.7,14.2,11.4,7.3,4.4,1.3,0.1,0.0,0.0,0.0
10,Hutnik Kraków,987.26,45.44,0.0,0.0,0.0,0.3,0.8,2.4,4.1,6.4,10.2,14.6,17.6,17.1,15.5,8.2,2.7,0.0,0.0,0.0


In [17]:
CONFIG['games_to_overwrite'][('Unia Skierniewice', 'Sokół Kleczew')] = (0, 2)
results, h2h_table = run_full_table_sims(**CONFIG, standings_df=standings_df, fixtures_matrix=fixtures_matrix)
results.head(CONFIG['head_size'])

100%|██████████| 5000/5000 [01:03<00:00, 78.51it/s]

5000 simulations


,Club,Elo,xPts,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18
1,Unia Skierniewice,1066.29,65.19,69.7,25.3,4.9,0.1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,Olimpia Grudziądz,1066.29,61.83,17.8,37.4,39.0,4.7,0.9,0.2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,Warta Poznań,1102.42,61.26,12.4,35.9,43.7,6.0,1.6,0.3,0.1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,Śląsk Wrocław II,1012.10,53.41,0.0,0.3,3.8,28.6,24.5,17.2,12.1,7.8,3.9,1.5,0.3,0.1,0.0,0.0,0.0,0.0,0.0,0.0
5,Podhale Nowy Targ,1021.13,52.90,0.1,0.5,3.9,21.8,21.4,17.9,13.2,9.8,6.4,3.3,1.3,0.4,0.1,0.0,0.0,0.0,0.0,0.0
6,Podbeskidzie,1023.39,52.04,0.0,0.1,1.1,16.6,19.5,20.4,16.2,11.5,7.8,3.7,1.9,1.0,0.3,0.0,0.0,0.0,0.0,0.0
7,Sandecja Nowy Sącz,998.55,51.50,0.0,0.4,2.9,13.3,15.0,17.3,17.7,14.4,10.3,5.5,2.2,0.7,0.3,0.1,0.0,0.0,0.0,0.0
8,Chojniczanka Chojnice,1059.52,49.82,0.0,0.0,0.5,4.5,9.0,12.7,17.9,18.8,14.9,10.2,5.6,3.1,2.1,0.7,0.0,0.0,0.0,0.0
9,Świt Skolwin,1030.16,48.45,0.0,0.0,0.3,4.0,6.1,8.4,11.2,13.6,15.0,13.3,11.4,8.7,5.2,2.8,0.3,0.0,0.0,0.0
10,Hutnik Kraków,987.26,45.42,0.0,0.0,0.0,0.3,1.0,2.1,4.1,6.5,9.7,13.5,14.8,15.8,15.8,12.8,3.6,0.0,0.0,0.0
